In [4]:
import pandas as pd
from google.colab import files
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# =========================
# UPLOAD DATASET
# =========================
print("="*70)
print("CANCER PREDICTION SYSTEM")
print("="*70)

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name, skipinitialspace=True, encoding='utf-8-sig')

# =========================
# DATA CLEANING
# =========================
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace("\ufeff", "")

for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.strip()

df = df.replace(['NA', 'nan', 'NaN', 'null', 'NULL'], np.nan)

for col in ['Age', 'WBC', 'RBC', 'Platelets', 'Haemoglobin', 'PSA', 'CA125', 'Family_History', 'Cancer']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# =========================
# ENCODE CANCER TYPE AND STAGE
# =========================
le_type = LabelEncoder()
le_stage = LabelEncoder()

if 'Cancer_Type' in df.columns:
    df['Cancer_Type_Encoded'] = le_type.fit_transform(df['Cancer_Type'].fillna('Unknown'))
    print(f"\n📊 Cancer Types: {list(le_type.classes_)}")

if 'Cancer_Stage' in df.columns:
    df['Cancer_Stage_Encoded'] = le_stage.fit_transform(df['Cancer_Stage'].fillna('Unknown'))
    print(f"📊 Cancer Stages: {list(le_stage.classes_)}")

# =========================
# FEATURES
# =========================
features = ['Age', 'WBC', 'RBC', 'Platelets', 'Haemoglobin', 'PSA', 'CA125', 'Family_History']

X = df[features]
y = df['Cancer']

for col in features:
    X[col] = X[col].fillna(X[col].median())

# =========================
# SCALE FEATURES
# =========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# =========================
# TRAIN CANCER DETECTION MODELS
# =========================

# 1. Logistic Regression
model_lr = LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000, random_state=42)
model_lr.fit(X_scaled, y)

# 2. SVM
model_svm = SVC(kernel='rbf', C=0.5, gamma='scale', class_weight='balanced',
                probability=True, random_state=42)
model_svm.fit(X_scaled, y)

# 3. Random Forest
model_rf = RandomForestClassifier(
    n_estimators=30, max_depth=3, min_samples_split=10,
    min_samples_leaf=5, class_weight='balanced', random_state=42
)
model_rf.fit(X_scaled, y)

# 4. XGBoost
model_xgb = XGBClassifier(
    n_estimators=30, max_depth=2, learning_rate=0.05,
    subsample=0.7, colsample_bytree=0.7, reg_alpha=0.5,
    reg_lambda=1.0, random_state=42, eval_metric='logloss',
    use_label_encoder=False
)
model_xgb.fit(X_scaled, y)

# =========================
# TRAIN CANCER TYPE MODEL (Only on cancer patients)
# =========================
model_type = None
if 'Cancer_Type_Encoded' in df.columns:
    df_cancer = df[df['Cancer'] == 1].copy()
    if len(df_cancer) >= 10:
        X_type = df_cancer[features]
        y_type = df_cancer['Cancer_Type_Encoded']

        for col in features:
            X_type[col] = X_type[col].fillna(X_type[col].median())

        X_type_scaled = scaler.transform(X_type)

        model_type = RandomForestClassifier(
            n_estimators=30, max_depth=3, min_samples_split=5,
            class_weight='balanced', random_state=42
        )
        model_type.fit(X_type_scaled, y_type)

# =========================
# TRAIN CANCER STAGE MODEL (Only on cancer patients)
# =========================
model_stage = None
if 'Cancer_Stage_Encoded' in df.columns:
    df_cancer = df[df['Cancer'] == 1].copy()
    if len(df_cancer) >= 10:
        X_stage = df_cancer[features]
        y_stage = df_cancer['Cancer_Stage_Encoded']

        for col in features:
            X_stage[col] = X_stage[col].fillna(X_stage[col].median())

        X_stage_scaled = scaler.transform(X_stage)

        model_stage = RandomForestClassifier(
            n_estimators=30, max_depth=3, min_samples_split=5,
            class_weight='balanced', random_state=42
        )
        model_stage.fit(X_stage_scaled, y_stage)

# =========================
# PREDICTION FUNCTION
# =========================
def predict_cancer(features_list):
    features_array = np.array(features_list).reshape(1, -1)
    features_scaled = scaler.transform(features_array)

    # Cancer detection
    proba_lr = model_lr.predict_proba(features_scaled)[0, 1]
    proba_svm = model_svm.predict_proba(features_scaled)[0, 1]
    proba_rf = model_rf.predict_proba(features_scaled)[0, 1]
    proba_xgb = model_xgb.predict_proba(features_scaled)[0, 1]

    cancer_proba = (proba_lr + proba_svm + proba_rf + proba_xgb) / 4
    has_cancer = cancer_proba > 0.5

    result = {
        'has_cancer': bool(has_cancer),
        'probability': float(cancer_proba)
    }

    # Cancer type prediction (if cancer detected)
    if has_cancer and model_type is not None:
        type_pred = model_type.predict(features_scaled)[0]
        type_proba = model_type.predict_proba(features_scaled)[0].max()
        result['cancer_type'] = le_type.inverse_transform([type_pred])[0]
        result['type_probability'] = float(type_proba)

    # Cancer stage prediction (if cancer detected)
    if has_cancer and model_stage is not None:
        stage_pred = model_stage.predict(features_scaled)[0]
        stage_proba = model_stage.predict_proba(features_scaled)[0].max()
        result['cancer_stage'] = le_stage.inverse_transform([stage_pred])[0]
        result['stage_probability'] = float(stage_proba)

    return result

# =========================
# INTERACTIVE INPUT
# =========================
print("\n" + "="*70)
print("ENTER PATIENT DETAILS")
print("="*70)

while True:
    print("\n" + "-"*50)

    try:
        age = float(input("Age: "))
        wbc = float(input("WBC: "))
        rbc = float(input("RBC: "))
        platelets = float(input("Platelets: "))
        haemoglobin = float(input("Haemoglobin: "))
        psa = float(input("PSA: "))
        ca125 = float(input("CA125: "))
        family_history = int(input("Family History (0/1): "))

        result = predict_cancer([age, wbc, rbc, platelets, haemoglobin, psa, ca125, family_history])

        print("\n" + "="*70)
        print("RESULT")
        print("="*70)

        if result['has_cancer']:
            print(f"\n⚠️ CANCER DETECTED")
            print(f"   Probability: {result['probability']:.1%}")

            if 'cancer_type' in result:
                print(f"\n🎯 CANCER TYPE")
                print(f"   Type: {result['cancer_type']}")
                print(f"   Confidence: {result['type_probability']:.1%}")

            if 'cancer_stage' in result:
                print(f"\n📊 CANCER STAGE")
                print(f"   Stage: {result['cancer_stage']}")
                print(f"   Confidence: {result['stage_probability']:.1%}")
        else:
            print(f"\n✅ NO CANCER DETECTED")
            print(f"   Probability: {result['probability']:.1%}")

        print("\n" + "="*70)

        again = input("\nAnother patient? (yes/no): ").strip().lower()
        if again != 'yes':
            break

    except ValueError:
        print("\n❌ Invalid input. Please enter numbers only.")
        continue
    except Exception as e:
        print(f"\n❌ Error: {e}")
        continue

print("\n✅ System ready.")

CANCER PREDICTION SYSTEM


Saving cancer_data.csv to cancer_data (2).csv

📊 Cancer Types: ['Breast', 'Colorectal', 'Lung', 'Pancreatic', 'Prostate', 'Unknown']
📊 Cancer Stages: ['Stage1', 'Stage2', 'Stage3', 'Stage4', 'Unknown']

ENTER PATIENT DETAILS

--------------------------------------------------
Age: 68
WBC: 7800
RBC: 4.9
Platelets: 310000
Haemoglobin: 14.2
PSA: 1.2
CA125: 85
Family History (0/1): 1

RESULT

⚠️ CANCER DETECTED
   Probability: 62.3%

🎯 CANCER TYPE
   Type: Prostate
   Confidence: 40.0%

📊 CANCER STAGE
   Stage: Stage3
   Confidence: 35.2%


Another patient? (yes/no): no

✅ System ready.
